# Baseline and Benchmark Analysis

This notebook analyzes existing benchmark artifacts only. It does not rerun benchmarks and does not call OpenAI, Pinecone, or local LoRA models.

In [ ]:
from pathlib import Path
import sys


def find_project_root(start: Path | None = None) -> Path:
    """Find the repository root from either repo-root or notebooks/ execution."""
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data" / "SMSSpamCollection").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("Could not find project root containing data/SMSSpamCollection and src/.")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_PATH = PROJECT_ROOT / "data" / "SMSSpamCollection"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
print(f"Project root: {PROJECT_ROOT.relative_to(PROJECT_ROOT)} (resolved internally)")

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, Markdown, display

plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True

RESULTS_PATH = OUTPUTS_DIR / "benchmark_results.csv"
PREDICTIONS_PATH = OUTPUTS_DIR / "benchmark_predictions.csv"
SUMMARY_PATH = OUTPUTS_DIR / "benchmark_summary.md"
CONFUSION_PATH = OUTPUTS_DIR / "confusion_matrices.png"

required = [RESULTS_PATH, PREDICTIONS_PATH, SUMMARY_PATH]
missing = [path.relative_to(PROJECT_ROOT) for path in required if not path.exists()]
HAVE_FILES = not missing
if missing:
    missing_text = ", ".join(str(p) for p in missing)
    display(Markdown(f"""**Missing benchmark artifacts.** Run this from the repo root before using this notebook:

```bash
python -m src.benchmark --modes tfidf_lr,base_llm,basic_rag,advanced_base,advanced_lora,guarded_fallback --sample-size 75
```

Missing: {missing_text}"""))
else:
    print("Benchmark artifacts found.")

## Benchmark Summary File

The generated Markdown summary is displayed first so the notebook mirrors the CLI report.

In [ ]:
if HAVE_FILES:
    display(Markdown(SUMMARY_PATH.read_text(encoding="utf-8")))

## Results Table

The CSV table is the source of truth for metrics and plotting.

In [ ]:
if HAVE_FILES:
    results = pd.read_csv(RESULTS_PATH)
    predictions = pd.read_csv(PREDICTIONS_PATH)
    display(results)
    print(f"Prediction rows: {len(predictions):,}")

## Official 75-Sample Check

This section verifies that the promoted official benchmark artifacts contain 75 samples per mode, 10 spam examples, and the expected LoRA/fallback behavior.

In [ ]:
if HAVE_FILES:
    tfidf_rows = predictions[predictions["mode"] == "tfidf_lr"]
    lora_row = results[results["mode"] == "advanced_lora"].iloc[0]
    guarded_row = results[results["mode"] == "guarded_fallback"].iloc[0]
    print(f"Samples per mode: {int(results['n_samples'].iloc[0])}")
    print(f"Spam examples: {int((tfidf_rows['true_label'] == 'spam').sum())}")
    print(f"advanced_lora spam_recall: {lora_row['spam_recall']:.4f}")
    print(f"guarded_fallback spam_recall: {guarded_row['spam_recall']:.4f}")

## Accuracy, Macro F1, and Spam Recall

These plots compare overall correctness, class-balanced F1, and the spam-specific recall that matters most for missed spam.

In [ ]:
if HAVE_FILES:
    metric_cols = ["accuracy", "f1_macro", "spam_recall"]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
    for ax, metric in zip(axes, metric_cols):
        ordered = results.sort_values(metric, ascending=False)
        ax.bar(ordered["mode"], ordered[metric], color="#4C78A8")
        ax.set_title(metric.replace("_", " ").title())
        ax.set_ylim(0, 1.05)
        ax.tick_params(axis="x", rotation=45)
    plt.tight_layout()
    plt.show()

TF-IDF is a strong classical baseline. The base LLM is weaker without retrieval, while Basic RAG and Advanced Base perform strongly on this sample. Advanced LoRA fails on spam recall, and Guarded Fallback recovers those LoRA failures.

## Average Latency by Mode

Latency is part of the system trade-off: the classical baseline is fastest, while agentic modes do more orchestration.

In [ ]:
if HAVE_FILES:
    ordered = results.sort_values("avg_latency_s")
    ax = ordered.plot(kind="bar", x="mode", y="avg_latency_s", legend=False, color="#72B7B2")
    ax.set_title("Average Latency by Mode")
    ax.set_xlabel("Mode")
    ax.set_ylabel("Seconds")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

The speed-quality trade-off is visible: TF-IDF is extremely fast and competitive, while retrieval and verifier/fallback logic add latency in exchange for stronger grounded behavior.

## Confusion Matrices

If the Phase 3 image exists, display it directly instead of regenerating plots.

In [ ]:
if HAVE_FILES:
    if CONFUSION_PATH.exists():
        display(Image(filename=str(CONFUSION_PATH)))
    else:
        display(Markdown("`outputs/confusion_matrices.png` was not found. The CSV metrics above are still available."))

## Takeaways

- TF-IDF is a strong, fast classical baseline.
- Base LLM is weaker without retrieval grounding.
- Basic RAG and Advanced Base perform strongly on this benchmark sample.
- Advanced LoRA is locally integrated but misses spam in this run.
- Guarded Fallback provides the reliability improvement by falling back when LoRA output is unsupported.